<a href="https://colab.research.google.com/github/Perry-wang-ab741/114-2-ProgramingLanguage/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [25]:
!pip install -q google-generativeai

In [26]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [27]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

In [28]:
from datetime import datetime

# Global variables _gc and _ws are managed by setup_gspread
_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        # client and MODEL_ID are globally defined in cell 81ade069
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        # SHEET_URL and WORKSHEET_NAME are globally defined or defined in ChVtDsoJ4F-R
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    try:
        # 尋找第一行空白列來寫入 AI 摘要
        next_row = len(_ws.col_values(1)) + 1
        _ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            # 確保不會寫入太長導致超出單元格限制 (雖然 gspread 會自動換行)
            _ws.update_cell(next_row + i, 3, line)
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [29]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [30]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


### 定義 AI 摘要函式

In [32]:
# ### AI 摘要函式已合併
#
# # `get_ai_summary` 函式已合併至 `process_grades_and_summary` 函式中，因此不再需要獨立執行此儲存格。

In [33]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

def process_grades_and_summary(grade_data):
    """
    處理 Gradio 介面傳入的成績，寫入 Google Sheet 並生成 AI 摘要。
    grade_data 預期是 [科目, 成績] 的列表的列表，例如：[['國文', 90], ['英文', 85]]
    """
    global _gc, _ws

    if _ws is None:
        # 如果連線失敗，嘗試重新設定 (可能在 Gradio 介面啟動後才執行)
        setup_gspread(SHEET_URL, WORKSHEET_NAME) # Modified to pass arguments
        if _ws is None:
            return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", ""

    if not grade_data:
        return "沒有輸入任何成績，請輸入科目和成績。", ""

    # 準備寫入 Google Sheet 的成績資料，增加日期欄位
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade_str in grade_data:
        try:
            grade = int(grade_str)
            new_grades_for_sheet.append([today, subject, grade])
        except ValueError:
            return f"科目 '{subject}' 的成績 '{grade_str}' 無效，成績必須是數字。", ""

    try:
        # 將新成績寫入 Google Sheet
        _ws.append_rows(new_grades_for_sheet)
        sheet_message = "成績已成功寫入 Google Sheet。\n"
    except Exception as e:
        sheet_message = f"寫入 Google Sheet 失敗：{e}\n"
        print(f"寫入 Google Sheet 失敗：{e}")
        # 即使寫入失敗，仍嘗試生成 AI 摘要

    # 獲取 AI 摘要
    summary = get_ai_summary(new_grades_for_sheet)

    try:
        # 尋找第一行空白列來寫入 AI 摘要
        next_row = len(_ws.col_values(1)) + 1
        _ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        _ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            # 確保不會寫入太長導致超出單元格限制 (雖然 gspread 會自動換行)
            _ws.update_cell(next_row + i, 3, line)
        sheet_message += "AI 摘要已成功寫入 Google Sheet。"
    except Exception as e:
        sheet_message += f"寫入 AI 摘要到 Google Sheet 失敗：{e}"
        print(f"寫入 AI 摘要到 Google Sheet 失敗：{e}")

    return sheet_message, summary

In [34]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")

--- 正在進行 Google Sheet 身份驗證和連線... ---
--- Google Sheet 連線成功。---

--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程不涉及任何評分或個人判斷。

---

### 成績摘要

這份成績列表呈現了學生在 **2026年4月14日** 三個科目的表現：
*   **國文**：85分
*   **數學**：78分
*   **英文**：92分

這是一次針對三個核心科目的成績記錄，涵蓋的成績範圍從78分到92分。其中，英文科目獲得了最高分，而數學科目則為這次記錄中的最低分。

---

### 關於成績的常見迷思整理

以下是關於學生學業成績的一些常見誤解或迷思，這些是普遍存在的觀點，與上述特定學生的成績表現無關：

1.  **迷思：成績是衡量一個學生能力或智力的唯一指標。**
    *   **事實：** 成績通常反映學生在特定時間、特定考試或作業中的表現，以及對特定知識點的掌握程度。它無法完全衡量學生的創造力、解決問題能力、情商、溝通協作能力、實踐技能或學習熱情等多元智能和潛力。

2.  **迷思：單一科目的低分代表學生「不擅長」該科目，甚至永遠無法學好。**
    *   **事實：** 一次考試的成績可能受到多種因素影響，例如考試當天的身心狀態、題型不適應、特定單元學習瓶頸、甚至是命題難度等。它只是一個時間點的反映，不代表學生長期的學習潛力或能力。持續的努力和適當的學習策略調整，往往能帶來進步。

3.  **迷思：高分學生未來一定成功，低分學生未來一定失敗。**
    *   **事實：** 雖然良好的學業成績能為學生提供更多升學或就業的機會，但現實世界的成功與否，除了學術能力，更取決於多元化的綜合素質，如適應力、韌性、人際關係、解決問題能力、創新思維以及對社會的貢獻等。許多在校成績不突出的人，在社會上同樣能取得卓越成就。

4.  **迷思：成

定義 Gradio 處理函式

In [36]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請在下方的表格中輸入學生的科目和成績，然後點擊『送出』。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Row():
        with gr.Column():
            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=[["", ""]],
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="輸入科目與成績 (點擊最後一行可新增)"
            )
            submit_button = gr.Button("送出")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(label="AI 摘要", lines=15)

    submit_button.click(
        process_grades_and_summary,
        inputs=grade_input,
        outputs=[sheet_output, summary_output]
    )

demo.launch()

/tmp/ipykernel_31235/1183026095.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04366cbe5c90575e36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
